   ... formalising the concepts from the initial handoff into a clean C header...

   Based on the architecture we discussed...

---
1. `eval_compute_geometry`
   THE MATH ENGINE
   This function is reponsible for translating the raw 3D world coordinates
   (pulled from MuJoCo sites) into the measurable distances needed for scoring.
   It doesn't judge success; it only calculates the physical state.

   Inside, the C code will likely look inside a series of vector math operations:
   - VECTOR CONSTRUCTION: Calculate the vector representing the socket's internal
     axis by subtracting `socket_mouth` from `socket_bottom`. Calculate the
     plug's relative position by subtracting `socket_mouth` from `plug_tip`.
   - NORMALISATION: Normalise the socket axis vector so it has a length of 1.
   - AXIAL DEPTH (Dot Product): Calculate the dot product of the plug's relative
     position and the normalised socket axis. This yields the `axial_depth` (how
     far along the hole the plug has traveled).
   - LATERAL ERROR (Vector Rejection): Multiply the `axial_depth` by the 
     normalised socket axis, and subtract that result from the plug's relative
     position vector. The magnitude (length) of this resulting vector is your
     `lateral_error` (how off-center the plug is),
   - STATE UPDATE: Write these calculated values directly into 
     `score->axial_depth`, `score->lateral_error`, and `score->plug_port_length`.

2. `eval_score_trail`
   THE GRADING RUBRIC
   This function acts as the judge. It takes the physical state calculated
   by the first function and evaluates it against the thresholds defined in
   `EvalConfig`.

   Inside, you will likely see a cascade of `if/else` statements and basic
   arithmetic:
   - OUTCOME CLASSIFICATION: It will check if `score->lateral_error` is less
     than `config->lateral_tol_m`.
      - If it is, it checks if `score->axial_depth` is extremely close to 
        `socket_depth` (within `config->depth_tol_m`). If yes, 
        `score->partial_insertion = 1`.
   - TIER 1 SCORE (Validity): Assigns a base score (e.g., 1.0) simply because
     the simulation ran without fatal errors.
   - TIER 2 SCORE (Motion Quality): Calculates deductions based on 
     `score->duration` (compared to `config->max_duration_s`), `score->path_length`
     , and penalizes `score->retries`.
   - TIER 3 SCORE (Task Success): Assigns the heavy-weight points based on the
     classification step (e.g., high points for full insertion, scaled points
     based on depth for partial).
   - FINAL TALLY: Sums Tier 1, 2, and 3 into `score->total`.    



   `chmod 644` makes files readable by all but writable only by the owner,
   `chmod 700` creates a private directory, and 
   `chmod 600` protects sensitive files.